# Module 9 — 進階應用綜合實戰

> **對應程度**：融合前八模組，對應真實工程問題

本模組涵蓋：
1. 有限元素法入門（FEM）
2. 多體動力學
3. 計算流體力學前置（CFD）
4. 狀態空間與控制

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as sla
from scipy.signal import place_poles
from scipy.sparse.linalg import spsolve, cg
import time
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.physics_models import (
    fem_bar_1d, spring_mass_system, simulate_spring_mass,
    heat_conduction_2d, inverted_pendulum_linearized,
    controllability_matrix, observability_matrix
)
from src.linalg_utils import lu_decomposition
from src.visualizer import set_style, plot_temperature_field, plot_mode_shapes

set_style()
print('Module 9 載入完成！')

---
## 9.1 有限元素法入門（FEM）

全域剛度矩陣組裝 → 施加邊界條件 → 求解 $K\vec{u} = \vec{f}$

In [ ]:
# 3 根彈性桿件串聯
nodes = [0.0, 1.0, 2.0, 3.0]  # 4 個節點
elements = [(0, 1), (1, 2), (2, 3)]  # 3 個桿件
EA = [100.0, 200.0, 150.0]  # 各桿件 EA

# 左端固定，右端施力 10N
boundary = {0: 0.0}
loads = {3: 10.0}

u, stress, K_global = fem_bar_1d(nodes, elements, EA, boundary, loads)

print('全域剛度矩陣 K =')
print(np.round(K_global, 1))
print(f'\nK 對稱: {np.allclose(K_global, K_global.T)} ✓')
print(f'\n節點位移: {np.round(u, 6)}')
print(f'桿件軸力: {np.round(stress, 4)}')

# 解析解驗證：串聯桿件，各桿件承受相同力 F=10N
u_analytical = np.zeros(4)
for i in range(3):
    L = nodes[i+1] - nodes[i]
    u_analytical[i+1] = u_analytical[i] + 10.0 * L / EA[i]

print(f'\n解析解位移: {np.round(u_analytical, 6)}')
print(f'FEM vs 解析解一致: {np.allclose(u, u_analytical, atol=1e-10)} ✓')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(nodes, u, 'bo-', lw=2, ms=8, label='FEM')
ax1.plot(nodes, u_analytical, 'rx--', lw=2, ms=8, label='解析解')
ax1.set_xlabel('位置 (m)')
ax1.set_ylabel('位移 (m)')
ax1.set_title('1D FEM 節點位移')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 應力圖
for i, (ni, nj) in enumerate(elements):
    ax2.barh(i, stress[i], color='steelblue', alpha=0.7)
    ax2.text(stress[i]+0.1, i, f'{stress[i]:.2f} N', va='center')
ax2.set_xlabel('軸力 (N)')
ax2.set_ylabel('桿件')
ax2.set_title('桿件軸力')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# LU 分解重複使用：不同荷載工況
# 只用自由度部分
free = [1, 2, 3]
K_ff = K_global[np.ix_(free, free)]
L_lu, U_lu = lu_decomposition(K_ff)

load_cases = {
    '右端 10N': {3: 10.0},
    '中間 5N': {2: 5.0},
    '均佈 2N': {1: 2.0, 2: 2.0, 3: 2.0},
}

fig, ax = plt.subplots(figsize=(10, 6))
for name, load in load_cases.items():
    f = np.zeros(3)
    for node_idx, force in load.items():
        f[node_idx - 1] = force
    y = sla.solve_triangular(L_lu, f, lower=True)
    x = sla.solve_triangular(U_lu, y, lower=False)
    u_full = np.zeros(4)
    u_full[free] = x
    ax.plot(nodes, u_full, 'o-', lw=2, label=name)

ax.set_xlabel('位置 (m)')
ax.set_ylabel('位移 (m)')
ax.set_title('FEM: LU 分解解不同荷載工況')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 9.2 多體動力學

廣義特徵值問題：$K\vec{\phi} = \omega^2 M\vec{\phi}$

In [ ]:
# 3 自由度彈簧-質量系統
masses = [2.0, 3.0, 1.5]
springs = [(-1, 0, 200), (0, 1, 150), (1, 2, 100), (2, -1, 80)]

M, K, frequencies, modes = spring_mass_system(masses, springs)

print(f'固有頻率: {np.round(frequencies, 4)} Hz')
print(f'\n振型:')
for i in range(3):
    print(f'  Mode {i+1} ({frequencies[i]:.2f} Hz): {np.round(modes[:, i], 4)}')

# 振型視覺化
fig, axes = plot_mode_shapes(modes, frequencies, n_modes=3)
plt.show()

In [ ]:
# 模態疊加法：任意初始條件的時域響應
x0 = np.array([0.01, 0.0, 0.0])  # 初始位移
v0 = np.zeros(3)                   # 初始速度

t, x = simulate_spring_mass(M, K, x0, v0, (0, 2.0))

fig, ax = plt.subplots(figsize=(14, 5))
for i in range(3):
    ax.plot(t, x[i], lw=1.5, label=f'm{i+1} ({masses[i]} kg)')
ax.set_xlabel('時間 (s)')
ax.set_ylabel('位移 (m)')
ax.set_title('3-DOF 彈簧-質量系統自由振動')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 頻率響應函數（FRF）
freq_range = np.linspace(0.1, 20, 1000)  # Hz
omega_range = 2 * np.pi * freq_range

# H(ω) = (-ω²M + K)^{-1}
FRF_11 = np.zeros(len(omega_range), dtype=complex)
for i, omega in enumerate(omega_range):
    H = np.linalg.inv(-omega**2 * M + K)
    FRF_11[i] = H[0, 0]  # 力在 m1, 響應在 m1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
ax1.semilogy(freq_range, np.abs(FRF_11), 'b-', lw=1.5)
for f in frequencies:
    ax1.axvline(f, color='r', ls='--', alpha=0.5)
ax1.set_ylabel('|H₁₁(f)| (m/N)')
ax1.set_title('頻率響應函數 (FRF)')
ax1.grid(True, alpha=0.3)

ax2.plot(freq_range, np.angle(FRF_11, deg=True), 'b-', lw=1.5)
for f in frequencies:
    ax2.axvline(f, color='r', ls='--', alpha=0.5, label=f'{f:.1f} Hz' if f == frequencies[0] else None)
ax2.set_xlabel('頻率 (Hz)')
ax2.set_ylabel('相位 (°)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f'共振峰（紅線）出現在固有頻率: {np.round(frequencies, 2)} Hz ✓')

---
## 9.3 計算流體力學前置（CFD）

2D 穩態熱傳導（泊松方程）$\nabla^2\phi = f$

In [ ]:
# 2D 穩態熱傳導
T_bc = {'top': 100, 'bottom': 0, 'left': 50, 'right': 50}
X, Y, T = heat_conduction_2d(nx=30, ny=30, T_boundary=T_bc)

fig, ax = plt.subplots(figsize=(10, 8))
plot_temperature_field(X, Y, T, title='2D 穩態溫度場', ax=ax)
plt.show()

# 驗證邊界條件
print(f'頂部邊界溫度: {T[-1, 1:-1].mean():.1f}°C (目標 100°C)')
print(f'底部邊界溫度: {T[0, 1:-1].mean():.1f}°C (目標 0°C)')
print(f'左側邊界溫度: {T[1:-1, 0].mean():.1f}°C (目標 50°C)')
print(f'右側邊界溫度: {T[1:-1, -1].mean():.1f}°C (目標 50°C)')

In [ ]:
# 直接求解 vs 迭代求解（CG）比較
from scipy.sparse import diags as sp_diags

grid_sizes = [10, 20, 30, 50]
times_direct = []
times_cg = []

for n in grid_sizes:
    N = n * n
    dx = 1.0 / (n + 1)
    main = 4.0/dx**2 * np.ones(N)
    off1 = -1.0/dx**2 * np.ones(N-1)
    offn = -1.0/dx**2 * np.ones(N-n)
    for i in range(1, n):
        off1[i*n-1] = 0
    A = sp_diags([main, off1, off1, offn, offn], [0, -1, 1, -n, n], format='csr')
    b = np.random.default_rng(42).normal(size=N)

    t0 = time.perf_counter()
    spsolve(A, b)
    times_direct.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    cg(A, b, tol=1e-8)
    times_cg.append(time.perf_counter() - t0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot([n**2 for n in grid_sizes], times_direct, 'bo-', lw=2, label='直接求解 (spsolve)')
ax.plot([n**2 for n in grid_sizes], times_cg, 'rs-', lw=2, label='迭代求解 (CG)')
ax.set_xlabel('未知數數量 N')
ax.set_ylabel('時間 (秒)')
ax.set_title('直接 vs 迭代求解器比較')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 9.4 狀態空間與控制

$$\dot{\vec{x}} = A\vec{x} + B\vec{u}, \quad \vec{y} = C\vec{x} + D\vec{u}$$

In [ ]:
# 倒擺狀態空間模型
A, B, C, D = inverted_pendulum_linearized()

print('倒擺狀態空間模型:')
print(f'  狀態: [x, x_dot, theta, theta_dot]')
print(f'  A = \n{np.round(A, 4)}')
print(f'  B = {B.flatten()}')

# 可控性、可觀性
Ctrb = controllability_matrix(A, B)
Obsv = observability_matrix(A, C)

print(f'\n可控性矩陣秩: {np.linalg.matrix_rank(Ctrb)} (= n={A.shape[0]} → 可控 ✓)')
print(f'可觀性矩陣秩: {np.linalg.matrix_rank(Obsv)} (= n={A.shape[0]} → 可觀 ✓)')

# 開迴路特徵值
eigs_open = np.linalg.eigvals(A)
print(f'\n開迴路特徵值: {np.round(eigs_open, 4)}')
print(f'系統不穩定（有正實部特徵值）')

In [ ]:
# 極點配置控制器設計
desired_poles = np.array([-2, -3, -1.5+1.5j, -1.5-1.5j])
result = place_poles(A, B, desired_poles)
K_ctrl = result.gain_matrix

A_cl = A - B @ K_ctrl
eigs_cl = np.linalg.eigvals(A_cl)

print('閉迴路特徵值:')
for e in eigs_cl:
    print(f'  λ = {e:.4f}')
print(f'\n所有特徵值實部 < 0: {all(np.real(eigs_cl) < 0)} ✓')

# 閉迴路時域模擬
from scipy.integrate import solve_ivp

def closed_loop(t, x):
    u = -K_ctrl @ x.reshape(-1, 1)
    return (A @ x.reshape(-1, 1) + B @ u).flatten()

x0 = np.array([0, 0, 0.2, 0])  # 初始角度偏移 0.2 rad ≈ 11.5°
sol = solve_ivp(closed_loop, [0, 5], x0, max_step=0.01)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
labels = ['位移 x (m)', '速度 dx/dt (m/s)', '角度 θ (rad)', '角速度 dθ/dt (rad/s)']
for i in range(4):
    ax = axes[i//2, i%2]
    ax.plot(sol.t, sol.y[i], 'b-', lw=1.5)
    ax.axhline(0, color='r', ls='--', alpha=0.5)
    ax.set_xlabel('時間 (s)')
    ax.set_ylabel(labels[i])
    ax.set_title(labels[i])
    ax.grid(True, alpha=0.3)

plt.suptitle('倒擺閉迴路控制響應 (初始偏移 θ₀ = 0.2 rad)', fontsize=14)
plt.tight_layout()
plt.show()
print(f'最終角度: {sol.y[2, -1]:.6f} rad → 穩定到平衡點 ✓')

In [ ]:
# 極點圖總結
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(np.real(eigs_open), np.imag(eigs_open), s=200, c='red',
           marker='x', lw=3, label='開迴路', zorder=5)
ax.scatter(np.real(eigs_cl), np.imag(eigs_cl), s=200, c='blue',
           marker='o', lw=2, label='閉迴路', zorder=5)
# 箭頭連接
for eo, ec in zip(eigs_open, eigs_cl):
    ax.annotate('', xy=(np.real(ec), np.imag(ec)),
                xytext=(np.real(eo), np.imag(eo)),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))

ax.axvline(0, color='k', lw=1)
ax.axhline(0, color='k', lw=1)
ax.fill_betweenx([-5, 5], -10, 0, alpha=0.05, color='green')
ax.set_xlabel('Re(λ)')
ax.set_ylabel('Im(λ)')
ax.set_title('極點配置：所有極點移至左半平面')
ax.legend(fontsize=12)
ax.set_xlim(-5, 5)
ax.set_ylim(-3, 3)
ax.grid(True, alpha=0.3)
plt.show()

---
## Module 9 驗證總結

| 項目 | 驗證方式 | 結果 |
|------|----------|------|
| FEM 位移 | 與解析解比對 | ✓ |
| 剛度矩陣 | 對稱性 | ✓ |
| 固有頻率 | 廣義特徵值 | ✓ |
| FRF 共振峰 | 出現在固有頻率 | ✓ |
| 2D 熱傳導 | 邊界條件滿足 | ✓ |
| 可控性 | rank(C) = n | ✓ |
| 極點配置 | Re(λ_cl) < 0 | ✓ |
| 倒擺穩定 | θ → 0 | ✓ |

---

## 恭喜完成所有模組！

你已經學會了從向量基礎到工程應用的完整線性代數技能。